# Exploring the occupancy permit data (layer 24)

There is no data dictionary for this layer, so before writing the enrichment script I looked at the data to answer a few questions:

1. What does the layer contain, and how complete is it?
2. How should the 2026 date rule work ("applications or issued permits")?
3. Are the two permit types spelled as in the brief?
4. Which column identifies the utility, and how are the names written?
5. Which column identifies a permit?
6. How many locations does a permit have?

Each section ends with what I decided and why.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))  # so the notebook can import from enrichment/
from enrichment.rest_client import fetch_layer

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)

LAYER_24 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_APPS/DDOT_TOPS/MapServer/24"

## 1. First look

I download the whole layer with no filter, so I can see everything before deciding how to filter it.

In [2]:
permits = fetch_layer(LAYER_24)
print(f"{len(permits):,} rows, {permits.shape[1]} columns")

20,039 rows, 116 columns


In [3]:
# How full is each column? (share of rows that are not blank)
filled = permits.notna().mean().sort_values(ascending=False).round(3)
print("Columns filled on every row:", (filled == 1).sum())
print("Columns that are completely empty:", (filled == 0).sum())
filled.to_frame("share_filled")

Columns filled on every row: 39
Columns that are completely empty: 8


,share_filled
OBJECTID,1.000
MeterPaidYN,1.000
AddressLockedYN,1.000
WorkLocationAddressID,1.000
WorkLocationFullAddress,1.000
PermitteeName,1.000
WorkLocationWard,1.000
WorkLocationANC,1.000
WorkLocationSMD,1.000
WorkLocationZip,1.000


In [4]:
# The columns that matter for this task
key_columns = [
    "TrackingNumber", "PermitNumber", "EventTypeDescription", "StatusDescription",
    "ApplicationDate", "IssueDate", "ApplicantCompany", "PermitteeName", "OwnerName",
    "WorkLocationFullAddress", "WorkLocationAddressID", "XCoord", "YCoord", "WorkLocationWard",
]
permits[key_columns].head(5)

,TrackingNumber,PermitNumber,EventTypeDescription,StatusDescription,ApplicationDate,IssueDate,ApplicantCompany,PermitteeName,OwnerName,WorkLocationFullAddress,WorkLocationAddressID,XCoord,YCoord,WorkLocationWard
0,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1700 - 1799 BLOCK OF EAST CAPITOL STREET,1433118,401745.69,135773.22,7
1,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1720 - 1799 BLOCK OF C STREET NE,1433119,401784.36,136161.68,7
2,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1800 - 1819 BLOCK OF C STREET NE,1433120,401859.92,136161.83,7
3,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1800 - 1899 BLOCK OF EAST CAPITOL STREET,1433121,401897.81,135773.26,7
4,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1820 - 1899 BLOCK OF C STREET NE,1433122,401935.17,136161.85,7


In [5]:
# Date range of the whole layer
permits[["ApplicationDate", "IssueDate"]].agg(["min", "max", "count"])

,ApplicationDate,IssueDate
min,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00
max,2026-10-13 04:06:04-04:00,2026-10-13 04:06:30-04:00
count,20039,14173


**What section 1 shows**

- 20,039 rows. Each row is one permit *location*, not one permit.
- `TrackingNumber` is filled on every row. `PermitNumber` is only filled on 70.7% of rows, the same share as `IssueDate`. So a permit number seems to be given only once the permit is issued, and applications still in progress don't have one yet. **TrackingNumber is the safer permit ID.**
- `PermitteeName` and `ApplicantCompany` are always filled. `OwnerName` is filled on about two-thirds of rows. All three could hold the utility name (section 4).
- Every row has coordinates (`XCoord`, `YCoord`), so every location can be matched to a road.
- `RoadwaySegID` is always empty and `StreetSegID` almost always empty, so the layer can't give us the road directly. That's why the road lookup is needed.
- The layer is not a full history: applications go back only to June 2025. It looks like a rolling window of recent and current permits (section 2).